"""
Detector de Anomalías en Datasets MPU-6050
Identifica problemas de conexión, valores fuera de rango, y datos congelados
"""

In [ ]:
import numpy as np
import pandas as pd
import os
import glob
from pathlib import Path

In [ ]:
 class MPU6050AnomalyDetector:
    """Detecta anomalías en datasets del MPU-6050"""

    def __init__(self, accel_range=8, gyro_range=1000):
        """
        Args:
            accel_range: Rango del acelerómetro en G (8 para ±8G)
            gyro_range: Rango del giroscopio en °/s (1000 para ±1000°/s)
        """
        # Límites físicos (con margen de 10% para ruido)
        self.accel_max = accel_range * 9.81 * 1.1  # m/s²
        self.gyro_max = np.deg2rad(gyro_range) * 1.1  # rad/s

    def load_dataset(self, filepath):
        """Carga dataset desde archivo"""
        try:
            # Intentar con Time primero (archivo limpio)
            data = pd.read_csv(filepath, sep=r'\s+', header=None,
                             names=['Time', 'AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ'])

            # Verificar si Time está en primera columna
            if data['Time'].iloc[0] > 100:  # Si el primer Time es muy grande, está en última columna
                data = pd.read_csv(filepath, sep=r'\s+', header=None,
                                 names=['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ', 'Time'])

            return data
        except Exception as e:
            print(f"❌ Error cargando {filepath}: {e}")
            return None

    def detect_out_of_range(self, data):
        """Detecta valores fuera del rango físico del sensor"""
        issues = []

        # Verificar acelerómetro
        for axis in ['AccX', 'AccY', 'AccZ']:
            out_of_range = np.abs(data[axis]) > self.accel_max
            if out_of_range.any():
                count = out_of_range.sum()
                max_val = data[axis].abs().max()
                indices = np.where(out_of_range)[0]
                issues.append({
                    'type': 'OUT_OF_RANGE',
                    'sensor': 'Acelerómetro',
                    'axis': axis,
                    'count': count,
                    'max_value': max_val,
                    'limit': self.accel_max,
                    'indices': indices[:5].tolist()  # Primeras 5 ocurrencias
                })

        # Verificar giroscopio
        for axis in ['GyrX', 'GyrY', 'GyrZ']:
            out_of_range = np.abs(data[axis]) > self.gyro_max
            if out_of_range.any():
                count = out_of_range.sum()
                max_val = data[axis].abs().max()
                indices = np.where(out_of_range)[0]
                issues.append({
                    'type': 'OUT_OF_RANGE',
                    'sensor': 'Giroscopio',
                    'axis': axis,
                    'count': count,
                    'max_value': max_val,
                    'limit': self.gyro_max,
                    'indices': indices[:5].tolist()
                })

        return issues

    def detect_frozen_values(self, data, threshold=3):
        """Detecta valores congelados (mismo valor repetido)"""
        issues = []

        for col in ['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ']:
            # Buscar valores idénticos consecutivos
            diff = data[col].diff()
            frozen = (diff == 0)

            # Contar secuencias de valores congelados
            frozen_sequences = []
            count = 0
            start_idx = None

            for idx, is_frozen in enumerate(frozen):
                if is_frozen:
                    if count == 0:
                        start_idx = idx - 1
                    count += 1
                else:
                    if count >= threshold:
                        frozen_sequences.append({
                            'start': start_idx,
                            'length': count,
                            'value': data[col].iloc[start_idx]
                        })
                    count = 0

            # Verificar última secuencia
            if count >= threshold:
                frozen_sequences.append({
                    'start': start_idx,
                    'length': count,
                    'value': data[col].iloc[start_idx]
                })

            if frozen_sequences:
                issues.append({
                    'type': 'FROZEN_VALUES',
                    'axis': col,
                    'sequences': len(frozen_sequences),
                    'longest': max(seq['length'] for seq in frozen_sequences),
                    'examples': frozen_sequences[:3]  # Primeras 3 secuencias
                })

        return issues

    def detect_sudden_spikes(self, data, threshold_sigma=100):
        """Detecta picos repentinos (outliers estadísticos)"""
        issues = []

        for col in ['AccX', 'AccY', 'AccZ', 'GyrX', 'GyrY', 'GyrZ']:
            # Calcular diferencias entre muestras consecutivas
            diff = data[col].diff().abs()

            # Detectar outliers usando desviación estándar
            mean_diff = diff.mean()
            std_diff = diff.std()
            threshold = mean_diff + threshold_sigma * std_diff

            spikes = diff > threshold
            if spikes.any():
                count = spikes.sum()
                indices = np.where(spikes)[0]
                spike_values = diff[spikes].values

                issues.append({
                    'type': 'SUDDEN_SPIKE',
                    'axis': col,
                    'count': count,
                    'threshold': threshold,
                    'max_spike': spike_values.max(),
                    'indices': indices[:5].tolist()
                })

        return issues

    def detect_zero_gyro(self, data, threshold=5):
        """Detecta giroscopio congelado en cero (común después de fallas I2C)"""
        issues = []

        for col in ['GyrX', 'GyrY', 'GyrZ']:
            zeros = (data[col] == 0)
            zero_count = zeros.sum()

            if zero_count >= threshold:
                issues.append({
                    'type': 'ZERO_GYRO',
                    'axis': col,
                    'count': zero_count,
                    'percentage': (zero_count / len(data)) * 100
                })

        return issues

    def analyze_dataset(self, filepath):
        """Análisis completo de un dataset"""
        data = self.load_dataset(filepath)
        if data is None:
            return None

        report = {
            'filename': os.path.basename(filepath),
            'samples': len(data),
            'duration': (data['Time'].iloc[-1] - data['Time'].iloc[0]) / 1000,
            'issues': [],
            'status': 'OK'
        }

        # Ejecutar todas las detecciones
        report['issues'].extend(self.detect_out_of_range(data))
        report['issues'].extend(self.detect_frozen_values(data))
        report['issues'].extend(self.detect_sudden_spikes(data))
        report['issues'].extend(self.detect_zero_gyro(data))

        # Determinar estado general
        if len(report['issues']) == 0:
            report['status'] = '✅ OK'
        elif any(issue['type'] in ['OUT_OF_RANGE', 'FROZEN_VALUES'] for issue in report['issues']):
            report['status'] = '❌ CORRUPTO'
        else:
            report['status'] = '⚠️ SOSPECHOSO'

        return report

    def batch_analyze(self, directory, pattern='*.txt'):
        """Analiza múltiples datasets en un directorio"""
        files = glob.glob(os.path.join(directory, pattern))

        if not files:
            print(f"❌ No se encontraron archivos con patrón '{pattern}' en {directory}")
            return []

        print(f"\n{'='*70}")
        print(f"ANALIZANDO {len(files)} DATASETS")
        print(f"{'='*70}\n")

        reports = []
        ok_count = 0
        corrupt_count = 0
        suspicious_count = 0

        for filepath in sorted(files):
            report = self.analyze_dataset(filepath)
            if report:
                reports.append(report)

                # Contar por estado
                if report['status'] == '✅ OK':
                    ok_count += 1
                elif report['status'] == '❌ CORRUPTO':
                    corrupt_count += 1
                else:
                    suspicious_count += 1

                # Imprimir resumen
                print(f"{report['status']} {report['filename']}")
                print(f"   Muestras: {report['samples']} | Duración: {report['duration']:.1f}s")

                if report['issues']:
                    for issue in report['issues'][:3]:  # Mostrar primeros 3 problemas
                        if issue['type'] == 'OUT_OF_RANGE':
                            print(f"   ⚠️  {issue['sensor']} {issue['axis']}: {issue['count']} valores fuera de rango (max: {issue['max_value']:.2f})")
                        elif issue['type'] == 'FROZEN_VALUES':
                            print(f"   ⚠️  {issue['axis']}: {issue['sequences']} secuencias congeladas (máx: {issue['longest']} muestras)")
                        elif issue['type'] == 'SUDDEN_SPIKE':
                            print(f"   ⚠️  {issue['axis']}: {issue['count']} picos repentinos (máx: {issue['max_spike']:.2f})")
                        elif issue['type'] == 'ZERO_GYRO':
                            print(f"   ⚠️  {issue['axis']}: {issue['count']} ceros ({issue['percentage']:.1f}%)")
                print()

        # Resumen final
        print(f"{'='*70}")
        print(f"RESUMEN")
        print(f"{'='*70}")
        print(f"✅ OK:          {ok_count} datasets")
        print(f"⚠️  SOSPECHOSOS: {suspicious_count} datasets")
        print(f"❌ CORRUPTOS:   {corrupt_count} datasets")
        print(f"\nTotal analizado: {len(reports)} datasets")
        print(f"Porcentaje OK: {(ok_count/len(reports)*100):.1f}%")

        return reports

    def save_report(self, reports, output_file='dataset_quality_report.txt'):
        """Guarda reporte detallado en archivo"""
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("="*70 + "\n")
            f.write("REPORTE DE CALIDAD DE DATASETS MPU-6050\n")
            f.write("="*70 + "\n\n")

            for report in reports:
                f.write(f"\n{report['status']} {report['filename']}\n")
                f.write(f"Muestras: {report['samples']} | Duración: {report['duration']:.1f}s\n")

                if report['issues']:
                    f.write("Problemas detectados:\n")
                    for issue in report['issues']:
                        f.write(f"  - {issue}\n")
                f.write("\n")

        print(f"\n📄 Reporte detallado guardado en: {output_file}")

In [ ]:
# ============================================
# EJEMPLO DE USO
# ============================================

if __name__ == "__main__":

    # Crear detector
    detector = MPU6050AnomalyDetector(accel_range=8, gyro_range=1000)

    # OPCIÓN 1: Analizar un solo archivo
    #print("\n=== ANÁLISIS DE ARCHIVO ÚNICO ===")
    #report = detector.analyze_dataset('shimmy_dataset.txt')
    #if report:
     #  for issue in report['issues']:
       #     print(f"  {issue}")

    # OPCIÓN 2: Analizar todos los archivos en un directorio
    print("\n\n=== ANÁLISIS EN LOTE ===")
    reports = detector.batch_analyze(
        directory='./DATASETS',  # Cambiar por tu carpeta
        pattern='*.txt'
    )

    # Guardar reporte
    if reports:
        detector.save_report(reports)

    # Listar solo los datasets OK para usar en ML
    print("\n" + "="*70)
    print("DATASETS VÁLIDOS PARA ENTRENAR ML:")
    print("="*70)
    ok_datasets = [r for r in reports if r['status'] == '✅ OK']
    for r in ok_datasets:
        print(f"  ✅ {r['filename']}")



=== ANÁLISIS EN LOTE ===

ANALIZANDO 1 DATASETS

❌ CORRUPTO DATASET_REPOSO_CLEAN.txt
   Muestras: 2747 | Duración: 54.9s
   ⚠️  GyrX: 1 secuencias congeladas (máx: 3 muestras)
   ⚠️  GyrY: 1 secuencias congeladas (máx: 4 muestras)
   ⚠️  GyrZ: 17 secuencias congeladas (máx: 5 muestras)

RESUMEN
✅ OK:          0 datasets
⚠️  SOSPECHOSOS: 0 datasets
❌ CORRUPTOS:   1 datasets

Total analizado: 1 datasets
Porcentaje OK: 0.0%

📄 Reporte detallado guardado en: dataset_quality_report.txt

DATASETS VÁLIDOS PARA ENTRENAR ML:
